In [11]:
# librerias
import pandas as pd

# ruta del archivo csv
path = r"C:\Users\User\Desktop\Linea Cero\DATA ANALYTICS\PROYECTOS DATA\proyecto_dw_salud\raw\nacidos-vivos-registrados-en-la-republica-argentina-entre-los-anos-2005-2022.csv"

# lectura
df = pd.read_csv(path, encoding="latin-1", low_memory=False)

# informacion del dataframe - Data Profiling (entender el dataframe, ver estructura, tipos de datos, valores nulos, etc.)
#Dimensiones del dataset
df.shape

#Esto permitió identificar la cantidad total de registros y variables presentes en el archivo.

#Nombres de las columnas
df.columns

#Este paso permitió identificar las variables disponibles en el dataset.

#Primeros registros
df.head()

#Esto permitió visualizar ejemplos reales de registros y comprender cómo se organizaban los datos.

#Este proceso de exploración inicial se conoce como Data Profiling, y es una etapa fundamental antes de cualquier proceso de modelado o transformación de datos.
df.columns
df.head(5) # muestra las primeras 5 filas del dataframe 

#//////////////////////////////////////////////////////////////////////////////////////////////////

import numpy as np

df = df.copy()

# 1) Jurisdicción: hay un ID (98) con nombre nulo → lo completamos
df["jurisdicion_residencia_nombre"] = df["jurisdicion_residencia_nombre"].fillna("No especificado")

# 2) Tipo de parto: hay 1 fila nula → completar con "Desconocido"
df["tipo_de_parto_id"] = df["tipo_de_parto_id"].fillna(0).astype(int)
df["tipo_de_parto_nombre"] = df["tipo_de_parto_nombre"].fillna("Desconocido")

# 3) Asegurar tipos
df["anio"] = df["anio"].astype(int)
df["jurisdiccion_de_residencia_id"] = df["jurisdiccion_de_residencia_id"].astype(int)
df["sexo_id"] = df["sexo_id"].astype(int)
df["nacimientos_cantidad"] = df["nacimientos_cantidad"].astype(int)

df.isna().sum()



def build_dim(df, cols, dim_name):
    dim = df[cols].drop_duplicates().reset_index(drop=True)
    dim.insert(0, f"{dim_name}_key", range(1, len(dim) + 1))
    return dim

dim_tiempo = build_dim(df, ["anio"], "tiempo")
dim_jurisdiccion = build_dim(df, ["jurisdiccion_de_residencia_id", "jurisdicion_residencia_nombre"], "jurisdiccion")
dim_edad_madre = build_dim(df, ["edad_madre_grupo"], "edad_madre")
dim_instruccion = build_dim(df, ["instruccion_madre"], "instruccion")
dim_tipo_parto = build_dim(df, ["tipo_de_parto_id", "tipo_de_parto_nombre"], "tipo_parto")
dim_semana = build_dim(df, ["semana_gestacion"], "semana_gestacion")
dim_peso = build_dim(df, ["intervalo_peso_al_nacer"], "peso")
dim_sexo = build_dim(df, ["sexo_id", "Sexo"], "sexo")

[len(x) for x in [dim_tiempo, dim_jurisdiccion, dim_edad_madre, dim_instruccion, dim_tipo_parto, dim_semana, dim_peso, dim_sexo]]



fact = df.merge(dim_tiempo, on="anio") \
         .merge(dim_jurisdiccion, on=["jurisdiccion_de_residencia_id","jurisdicion_residencia_nombre"]) \
         .merge(dim_edad_madre, on="edad_madre_grupo") \
         .merge(dim_instruccion, on="instruccion_madre") \
         .merge(dim_tipo_parto, on=["tipo_de_parto_id","tipo_de_parto_nombre"]) \
         .merge(dim_semana, on="semana_gestacion") \
         .merge(dim_peso, on="intervalo_peso_al_nacer") \
         .merge(dim_sexo, on=["sexo_id","Sexo"])

fact_dw = fact[[
    "tiempo_key","jurisdiccion_key","edad_madre_key","instruccion_key","tipo_parto_key",
    "semana_gestacion_key","peso_key","sexo_key","nacimientos_cantidad"
]]

fact_dw.head()


out = r"C:\Users\User\Desktop\Linea Cero\DATA ANALYTICS\PROYECTOS DATA\proyecto_dw_salud\csv"
 
dim_tiempo.to_csv(out + r"\dim_tiempo.csv", index=False)
dim_jurisdiccion.to_csv(out + r"\dim_jurisdiccion.csv", index=False)
dim_edad_madre.to_csv(out + r"\dim_edad_madre.csv", index=False)
dim_instruccion.to_csv(out + r"\dim_instruccion.csv", index=False)
dim_tipo_parto.to_csv(out + r"\dim_tipo_parto.csv", index=False)
dim_semana.to_csv(out + r"\dim_semana_gestacion.csv", index=False)
dim_peso.to_csv(out + r"\dim_peso.csv", index=False)
dim_sexo.to_csv(out + r"\dim_sexo.csv", index=False)

fact_dw.to_csv(out + r"\fact_nacimientos.csv", index=False)

staging_df = df.rename(columns={
    "jurisdiccion_de_residencia_id": "jurisdiccion_id",
    "jurisdicion_residencia_nombre": "jurisdiccion_nombre",
    "instruccion_madre": "nivel_instruccion",
    "tipo_de_parto_nombre": "tipo_parto",
    "semana_gestacion": "semanas_gestacion",
    "intervalo_peso_al_nacer": "peso_rn",
    "Sexo": "sexo"
})[[
    "anio",
    "jurisdiccion_id",
    "jurisdiccion_nombre",
    "edad_madre_grupo",
    "nivel_instruccion",
    "tipo_parto",
    "semanas_gestacion",
    "peso_rn",
    "sexo",
    "nacimientos_cantidad"
]]

staging_df.head()


,anio,jurisdiccion_id,jurisdiccion_nombre,edad_madre_grupo,nivel_instruccion,tipo_parto,semanas_gestacion,peso_rn,sexo,nacimientos_cantidad
0,2005,2,Ciudad AutÃ³noma de Buenos Aires,5.30 a 34,3.Primaria/C. EGB Completa,Simple,4.28 a 31,2.500 a 999,masculino,2
1,2022,6,Buenos Aires,3.20 a 24,4.Secundaria/Polimodal Incompleta,Simple,1.Menos de 22,2.500 a 999,masculino,1
2,2005,2,Ciudad AutÃ³noma de Buenos Aires,5.30 a 34,3.Primaria/C. EGB Completa,Simple,3.24 a 27,2.500 a 999,masculino,2
3,2022,6,Buenos Aires,3.20 a 24,4.Secundaria/Polimodal Incompleta,Simple,3.24 a 27,4.1500 a 1999,masculino,2
4,2005,2,Ciudad AutÃ³noma de Buenos Aires,5.30 a 34,3.Primaria/C. EGB Completa,Simple,2.22 a 23,2.500 a 999,masculino,1
